## converting a single row to structured json format

In [2]:
import re
import json

ASSIGN_PATTERN = re.compile(
    r"(\w+)\s*=\s*([a-zA-Z0-9_]+)\("
)

DIRECT_CALL_PATTERN = re.compile(
    r"^\s*([a-zA-Z0-9_]+)\("
)

def workflow_to_graph(workflow_code):
    steps = []
    step_id = 1

    seen = set()

    for line in workflow_code.split("\n"):
        line = line.strip()

        if not line:
            continue

        if line.startswith("#"):
            continue

        # variable = function(...)
        m = ASSIGN_PATTERN.match(line)

        if m:
            output_var = m.group(1)
            function_name = m.group(2)

            steps.append({
                "id": step_id,
                "tool": function_name,
                "output": output_var
            })

            step_id += 1
            continue

        # direct function(...)
        m = DIRECT_CALL_PATTERN.match(line)

        if m:
            function_name = m.group(1)

            # avoid counting control statements
            if function_name in {
                "if", "for", "while", "match",
                "case", "return", "print"
            }:
                continue

            steps.append({
                "id": step_id,
                "tool": function_name,
                "output": None
            })

            step_id += 1

    return {"steps": steps}

In [12]:
import json

with open("workflowLLM_data.json","r",encoding="utf-8") as f:
    data = json.load(f)

print("Samples:", len(data))

dataset = data


Samples: 11033


In [13]:
3+4

7

In [14]:
graph = workflow_to_graph(dataset[0]["workflow_code"])

print("Steps:", len(graph["steps"]))

for step in graph["steps"][:20]:
    print(step)

Steps: 15
{'id': 1, 'tool': 'is_workflow_actions_weather_currentconditions', 'output': 'current_weather_conditions'}
{'id': 2, 'tool': 'is_workflow_actions_text_replace', 'output': 'formatted_temperature'}
{'id': 3, 'tool': 'is_workflow_actions_text_replace', 'output': 'formatted_windspeed'}
{'id': 4, 'tool': 'is_workflow_actions_speaktext', 'output': None}
{'id': 5, 'tool': 'is_workflow_actions_readinglist', 'output': None}
{'id': 6, 'tool': 'is_workflow_actions_get_playlist', 'output': 'playlist'}
{'id': 7, 'tool': 'is_workflow_actions_getnameofemoji', 'output': 'emoji_name'}
{'id': 8, 'tool': 'is_workflow_actions_speaktext', 'output': None}
{'id': 9, 'tool': 'is_workflow_actions_delay', 'output': None}
{'id': 10, 'tool': 'is_workflow_actions_speaktext', 'output': None}
{'id': 11, 'tool': 'is_workflow_actions_dictatetext', 'output': 'user_selection'}
{'id': 12, 'tool': 'is_workflow_actions_image_mask', 'output': 'masked_image'}
{'id': 13, 'tool': 'is_workflow_actions_setparkedcar', '

In [15]:
step_counts = [
    len(workflow_to_graph(x["workflow_code"])["steps"])
    for x in dataset
]

print("avg", sum(step_counts)/len(step_counts))
print("max", max(step_counts))
print("min", min(step_counts))

avg 12.495150910903652
max 47
min 0


In [16]:
# analyse the min = 0
failed = []

for i, sample in enumerate(dataset):
    graph = workflow_to_graph(sample["workflow_code"])

    if len(graph["steps"]) == 0:
        failed.append(i)

print("Failed samples:", len(failed))
print(failed[:20])


Failed samples: 80
[110, 222, 282, 735, 906, 911, 1472, 1817, 1851, 1876, 1967, 2191, 2594, 3022, 3088, 3100, 3162, 3257, 3365, 3481]


In [17]:
idx = failed[0]

print(dataset[idx]["query"])
print("-" * 80)
print(dataset[idx]["workflow_code"][:5000])

How can I create a personalized entertainment schedule that includes reminders for upcoming movie releases, allows me to open my favorite streaming apps directly, and generates a QR code for sharing my watchlist with friends? Additionally, I would like to set up notifications for new episodes of my favorite shows and have the ability to quickly access reviews and trailers for the movies I plan to watch.
--------------------------------------------------------------------------------
import datetime    
from PIL import Image        
import qrcode  

# Step 1: Define User Preferences
# Create a dictionary to store user preferences including favorite streaming apps and other details
user_preferences = {  
    "favorites": ["Netflix", "Disney+", "HBO Max", "Max Go", "Prime Video", "YouTube"],  
    "favorite_theater": "Regal",  
    "name": "",  
    "birthday": "1994-05-28",  
    "email": "",  
    "phone": "Unlisted",  
    "home": "Unlisted",  
    "work": "Unlisted",  
    "calendars"

In [18]:
workflow_count = 0
python_count = 0

for sample in dataset:

    code = sample["workflow_code"]

    if (
        "is_workflow_actions_" in code
        or "com_" in code
    ):
        workflow_count += 1
    else:
        python_count += 1

print("workflow:", workflow_count)
print("python:", python_count)

workflow: 9602
python: 1431


In [20]:
import re

API_PATTERN = re.compile(
    r"(is_workflow_actions_[A-Za-z0-9_]+|com_[A-Za-z0-9_]+)"
)

def extract_apis(code):
    return sorted(set(API_PATTERN.findall(code)))

In [21]:
api_counts = []

for sample in dataset:
    apis = extract_apis(sample["workflow_code"])

    if len(apis) > 0:
        api_counts.append(len(apis))

print("Samples:", len(api_counts))
print("Avg APIs:", sum(api_counts)/len(api_counts))
print("Max APIs:", max(api_counts))
print("Min APIs:", min(api_counts))

Samples: 9602
Avg APIs: 7.157883774213706
Max APIs: 32
Min APIs: 1


In [22]:
import re

API_PATTERN = re.compile(
    r"(is_workflow_actions_[A-Za-z0-9_]+|com_[A-Za-z0-9_]+)"
)

total_api_mentions = 0
total_extracted_steps = 0

for sample in dataset:

    code = sample["workflow_code"]

    api_mentions = len(API_PATTERN.findall(code))

    graph = workflow_to_graph(code)

    total_api_mentions += api_mentions
    total_extracted_steps += len(graph["steps"])

print("API mentions:", total_api_mentions)
print("Extracted steps:", total_extracted_steps)

print(
    "Coverage:",
    round(
        total_extracted_steps /
        total_api_mentions * 100,
        2
    ),
    "%"
)

API mentions: 102649
Extracted steps: 137859
Coverage: 134.3 %


In [23]:
graph = workflow_to_graph(sample["workflow_code"])

for step in graph["steps"][:50]:
    print(step["tool"])


com_apple_Pages_TSADocumentCreateIntent
is_workflow_actions_setvalueforkey
com_apple_Pages_TSADocumentOpenIntent
com_agiletortoise_Drafts5_GetDraftIntent
com_agiletortoise_Drafts5_LiveActivityDraftIntent
com_agiletortoise_Drafts5_CaptureIntent
com_brogrammers_charty_AddSeriesFromCSVIntent
com_brogrammers_charty_StyleRingSeriesIntent
is_workflow_actions_makegif
is_workflow_actions_sendmessage


In [24]:
stage2_dataset = []

for sample in dataset:

    apis = extract_apis(
        sample["workflow_code"]
    )

    if not apis:
        continue

    graph = workflow_to_graph(
        sample["workflow_code"]
    )

    stage2_dataset.append({
        "query": sample["query"],
        "task_plan": sample["task_plan"],
        "apis": apis,
        "workflow_graph": graph
    })

In [25]:
import json

with open(
    "stage2_dataset.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        stage2_dataset,
        f,
        indent=2,
        ensure_ascii=False
    )

In [26]:
import ast

def build_workflow_graph(code):
    tree = ast.parse(code)

    steps = []
    variable_producers = {}
    step_id = 1

    for node in ast.walk(tree):

        if not isinstance(node, ast.Assign):
            continue

        if not isinstance(node.value, ast.Call):
            continue

        call = node.value

        func_name = None

        if isinstance(call.func, ast.Name):
            func_name = call.func.id

        if func_name is None:
            continue

        if not (
            func_name.startswith("is_workflow_actions_")
            or func_name.startswith("com_")
        ):
            continue

        output_var = None

        if len(node.targets) == 1 and isinstance(node.targets[0], ast.Name):
            output_var = node.targets[0].id

        inputs = []
        depends_on = []

        for kw in call.keywords:

            value = kw.value

            if isinstance(value, ast.Name):

                var = value.id

                inputs.append(var)

                if var in variable_producers:
                    depends_on.append(
                        variable_producers[var]
                    )

        step = {
            "id": step_id,
            "tool": func_name,
            "inputs": inputs,
            "depends_on": sorted(list(set(depends_on))),
            "output": output_var
        }

        steps.append(step)

        if output_var:
            variable_producers[output_var] = step_id

        step_id += 1

    return {"steps": steps}

In [27]:
sample = dataset[0]

In [28]:
graph = build_workflow_graph(
    sample["workflow_code"]
)

print(graph)

{'steps': [{'id': 1, 'tool': 'is_workflow_actions_weather_currentconditions', 'inputs': [], 'depends_on': [], 'output': 'current_weather_conditions'}, {'id': 2, 'tool': 'is_workflow_actions_text_replace', 'inputs': [], 'depends_on': [], 'output': 'formatted_temperature'}, {'id': 3, 'tool': 'is_workflow_actions_text_replace', 'inputs': [], 'depends_on': [], 'output': 'formatted_windspeed'}, {'id': 4, 'tool': 'is_workflow_actions_get_playlist', 'inputs': [], 'depends_on': [], 'output': 'playlist'}, {'id': 5, 'tool': 'is_workflow_actions_getnameofemoji', 'inputs': [], 'depends_on': [], 'output': 'emoji_name'}, {'id': 6, 'tool': 'is_workflow_actions_dictatetext', 'inputs': [], 'depends_on': [], 'output': 'user_selection'}, {'id': 7, 'tool': 'is_workflow_actions_image_mask', 'inputs': ['user_selection'], 'depends_on': [6], 'output': 'masked_image'}, {'id': 8, 'tool': 'is_workflow_actions_setparkedcar', 'inputs': ['masked_image'], 'depends_on': [7], 'output': 'parked_car_location'}]}


In [29]:
stage2_dataset = []

for sample in data:

    try:

        graph = build_workflow_graph(
            sample["workflow_code"]
        )

        if len(graph["steps"]) == 0:
            continue

        stage2_dataset.append({
            "query": sample["query"],
            "task_plan": sample["task_plan"],
            "apis": sample["apis_used"],
            "workflow_graph": graph
        })

    except Exception:
        pass

<unknown>:22: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:36: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:42: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:25: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:37: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\s" is

In [31]:
import json

with open(
    "stage2_dataset2.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        stage2_dataset,
        f,
        indent=2,
        ensure_ascii=False
    )

print(len(stage2_dataset))

5588


In [32]:
import random
import pprint

for sample in random.sample(stage2_dataset, 10):

    print("=" * 80)
    print(sample["query"][:150])

    pprint.pp(
        sample["workflow_graph"]["steps"][:5]
    )

How can I create a workflow on my device that automatically retrieves and displays educational resources from my Pocket account, while also sending a 
[{'id': 1,
  'tool': 'is_workflow_actions_pocket_get',
  'inputs': [],
  'depends_on': [],
  'output': 'pocket_items'},
 {'id': 2,
  'tool': 'is_workflow_actions_choosefromlist',
  'inputs': ['pocket_items'],
  'depends_on': [1],
  'output': 'selected_pocket_items'},
 {'id': 3,
  'tool': 'is_workflow_actions_getitemfromlist',
  'inputs': ['selected_pocket_items'],
  'depends_on': [2],
  'output': 'selected_pocket_item'},
 {'id': 4,
  'tool': 'is_workflow_actions_url',
  'inputs': ['selected_pocket_item'],
  'depends_on': [3],
  'output': 'pocket_item_url'},
 {'id': 5,
  'tool': 'is_workflow_actions_url_expand',
  'inputs': ['pocket_item_url'],
  'depends_on': [4],
  'output': 'expanded_pocket_item_url'}]
How can I create a new mind map document with specific text, pin it for quick access, and then export it to a PDF format using Python? 

In [ ]:
# len(stage2_dataset[0]['apis'])!= 0,len(stage2_dataset[0]['workflow_graph']['steps']) >= 2

(487, 8)

In [45]:
stage2_dataset[0]

{'query': 'How can I create a script that retrieves the current weather conditions for a specific city, formats the temperature and wind speed into a user-friendly message, and then vocalizes this information while also providing an option to save the details to a reading list for future reference?',
 'task_plan': "1. **Start**: Initialize the workflow to retrieve weather.\n2. **Retrieve Current Weather Conditions**: Fetch current weather using the `is_workflow_actions_weather_currentconditions()` function.\n3. **Format Temperature**: Format the temperature string by replacing '°F' using `is_workflow_actions_text_replace()`.\n4. **Format Wind Speed**: Format the wind speed string by replacing 'mph' using `is_workflow_actions_text_replace()`.\n5. **Create Weather Report**: Generate a user-friendly message that includes city name, temperature, weather condition, and wind speed.\n6. **Speak Weather Report**: Use `is_workflow_actions_speaktext()` to vocalize the weather report.\n7. **User 

In [ ]:
len(stage2_dataset[0]['apis'])!= 0,len(stage2_dataset[0]['workflow_graph']['steps']) >= 2

In [53]:
import json

# 1. Read the JSON file
# with open(r'C:\Users\veena\Desktop\manipal\mini_project\stage2_dataset2.json', 'r') as file:
#     data = json.load(file)

# 2. Filter out the row (e.g., removing the row where id is 2)
updated_data = [row for row in stage2_dataset if (len(row['apis']) != 0) and (len(row['workflow_graph']['steps']) >= 2)]

# 3. Save the changes back to the file
# with open('stage2_filtered_data.json', 'w') as file:
#     json.dump(updated_data, file, indent=4)


In [54]:
len(updated_data)

4571

In [56]:
updated_data = []

for row in stage2_dataset:
    apis = row["apis"]
    steps = row["workflow_graph"]["steps"]

    if len(apis) == 0:
        continue

    if len(steps) < 2:
        continue

    if len(steps) > 30:
        continue

    
print(len(updated_data))

0


In [ ]:
# # 3. Save the changes back to the file
# with open('stage2_filtered_data.json', 'w') as file:
#     json.dump(updated_data, file, indent=4)

#### Sanity check

In [69]:
# 1. Read the JSON file
with open(r'stage2_filtered_data.json', 'r') as file:
    filtered_data = json.load(file)

In [60]:
step_counts = [
    len(x["workflow_graph"]["steps"])
    for x in filtered_data
]

print("avg =", sum(step_counts)/len(step_counts))
print("max =", max(step_counts))
print("min =", min(step_counts))

avg = 6.807481951432947
max = 29
min = 2


sft fromat

In [61]:
import json

with open("stage2_filtered_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

with open("stage2_train.jsonl", "w", encoding="utf-8") as out:

    for sample in data:

        api_text = "\n".join(sample["apis"])

        user_prompt = f"""User Query:
{sample['query']}

Available APIs:
{api_text}
"""

        assistant_response = json.dumps(
            sample["workflow_graph"],
            ensure_ascii=False
        )

        record = {
            "messages": [
                {
                    "role": "user",
                    "content": user_prompt
                },
                {
                    "role": "assistant",
                    "content": assistant_response
                }
            ]
        }

        out.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Done")

Done


In [71]:
merge_nodes = 0
total_steps = 0

for row in filtered_data:
    for step in row["workflow_graph"]["steps"]:
        total_steps += 1

        if len(step.get("depends_on", [])) > 1:
            merge_nodes += 1

print("Merge nodes:", merge_nodes)
print("Total steps:", total_steps)
print("Percent:", merge_nodes / total_steps * 100)

Merge nodes: 335
Total steps: 31117
Percent: 1.076581932705595


In [73]:
terminal = 0

for row in filtered_data:
    last = row["workflow_graph"]["steps"][-1]["tool"]

    if any(x in last.lower() for x in [
        "save",
        "export",
        "send",
        "email",
        "share"
    ]):
        terminal += 1

print(terminal / len(dataset))

0.03000090637179371


In [62]:
import json

cnt = 0

with open("stage2_train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        cnt += 1

print("Samples:", cnt)

Samples: 4571


In [65]:
sample['messages']

[{'role': 'user',
  'content': 'User Query:\nHow can I create a script that retrieves the current weather conditions for a specific city, formats the temperature and wind speed into a user-friendly message, and then vocalizes this information while also providing an option to save the details to a reading list for future reference?\n\nAvailable APIs:\n[\n"\ni\ns\n_\nw\no\nr\nk\nf\nl\no\nw\n_\na\nc\nt\ni\no\nn\ns\n_\nd\ni\nc\nt\na\nt\ne\nt\ne\nx\nt\n"\n,\n \n"\ni\ns\n_\nw\no\nr\nk\nf\nl\no\nw\n_\na\nc\nt\ni\no\nn\ns\n_\nt\ne\nx\nt\n_\nr\ne\np\nl\na\nc\ne\n"\n,\n \n"\ni\ns\n_\nw\no\nr\nk\nf\nl\no\nw\n_\na\nc\nt\ni\no\nn\ns\n_\nr\ne\na\nd\ni\nn\ng\nl\ni\ns\nt\n"\n,\n \n"\ni\ns\n_\nw\no\nr\nk\nf\nl\no\nw\n_\na\nc\nt\ni\no\nn\ns\n_\ng\ne\nt\n_\np\nl\na\ny\nl\ni\ns\nt\n"\n,\n \n"\nc\no\nm\n_\ng\ne\nt\nc\na\nr\nd\np\no\ni\nn\nt\ne\nr\ns\n_\na\np\np\n_\nI\nn\nt\ne\nr\na\nc\nt\ni\nv\ne\nP\no\ni\nn\nt\ne\nr\ns\nC\no\nn\nf\ni\ng\nu\nr\na\nt\ni\no\nn\nA\np\np\nI\nn\nt\ne\nn\nt\n"\n,\n \n"\ni\ns\n_

In [67]:
import json

with open("stage2_train.jsonl", "r", encoding="utf-8") as f:
    sample = json.loads(next(f))

print(rf"{sample["messages"][0]["content"][:500]}")
print()
print(rf"{sample["messages"][1]["content"][:1000]}")

User Query:
How can I create a script that retrieves the current weather conditions for a specific city, formats the temperature and wind speed into a user-friendly message, and then vocalizes this information while also providing an option to save the details to a reading list for future reference?

Available APIs:
[
"
i
s
_
w
o
r
k
f
l
o
w
_
a
c
t
i
o
n
s
_
d
i
c
t
a
t
e
t
e
x
t
"
,
 
"
i
s
_
w
o
r
k
f
l
o
w
_
a
c
t
i
o
n
s
_
t
e
x
t
_
r
e
p
l
a
c
e
"
,
 
"
i
s
_
w
o
r
k
f
l
o
w
_
a
c
t
i
o
n


{"steps": [{"id": 1, "tool": "is_workflow_actions_weather_currentconditions", "inputs": [], "depends_on": [], "output": "current_weather_conditions"}, {"id": 2, "tool": "is_workflow_actions_text_replace", "inputs": [], "depends_on": [], "output": "formatted_temperature"}, {"id": 3, "tool": "is_workflow_actions_text_replace", "inputs": [], "depends_on": [], "output": "formatted_windspeed"}, {"id": 4, "tool": "is_workflow_actions_get_playlist", "inputs": [], "depends_on": [], "output": "playlist